# MSc. Financial Markets & Technologies
## Data Visualization Session 2 - Work with data

Let's begin importing libraries

In [6]:
import numpy as np
import pandas as pd
from pandas import Series, DataFrame, read_html

import urllib, json
from urllib.request import urlopen

from random import gauss, seed, randint

from datetime import datetime as dt

Reading CSV files

In [10]:
btc = pd.read_csv("BTC-USD.csv")
btc.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,2014-09-17,465.864014,468.174011,452.421997,457.334015,457.334015,21056800
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,424.440002,34483200
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,394.795990,37919700
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,408.903992,36863600
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,398.821014,26580100


Reading a Excel file

In [12]:
file="BTC-USD.xlsx"
sheet="Sheet1"

xl=pd.ExcelFile(file)
btc_xl=xl.parse(sheet)

btc_xl.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,2014-09-17,465.864014,468.174011,452.421997,457.334015,457.334015,21056800
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,424.440002,34483200
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,394.795990,37919700
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,408.903992,36863600
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,398.821014,26580100


Getting data from a webpage

In [13]:
url="https://finance.yahoo.com/quote/BTC-USD/history?p=BTC-USD"
btc_html=pd.io.html.read_html(url)[0]

btc_html.head()

,Date,Open,High,Low,Close*,Adj Close**,Volume
0,"Oct 20, 2020",11057.01,11743.26,11057.01,11743.26,11743.26,26571210752
1,"Oct 19, 2020",11495.04,11799.09,11408.29,11742.04,11742.04,23860769928
2,"Oct 18, 2020",11355.98,11483.36,11347.58,11483.36,11483.36,18283314340
3,"Oct 17, 2020",11322.12,11386.26,11285.35,11358.10,11358.10,19130430174
4,"Oct 16, 2020",11502.83,11540.06,11223.01,11322.12,11322.12,25635480772


We get a much more limited dataset through the web

In [14]:
print(len(btc))
print(len(btc_html))

1949
101


## Handling Missing Data

In [16]:
df = pd.DataFrame([[1, np.nan, 2],
                 [2,3,5],
                 [np.nan, 4,6]])

df

,0,1,2
0,1.0,NaN,2
1,2.0,3.0,5
2,NaN,4.0,6


There is a dropna() function, but it's a bit harsh or too restrictive

In [17]:
df.dropna()

,0,1,2
1,2.0,3.0,5


Let's add an empty column

In [18]:
df[3]=np.nan

df

,0,1,2,3
0,1.0,NaN,2,NaN
1,2.0,3.0,5,NaN
2,NaN,4.0,6,NaN


In [19]:
df.dropna()

,0,1,2,3


There is a parameter to indicate on which axis you want to drop values

In [20]:
df.dropna(axis='columns')

,2
0,2
1,5
2,6


There is another parameter, "how" which is quite usefull

In [21]:
df.dropna(axis='columns', how='all')

,0,1,2
0,1.0,NaN,2
1,2.0,3.0,5
2,NaN,4.0,6


The last parameter that we are going to see is the "threshold"

In [23]:
df.dropna(axis='rows', thresh=3)

,0,1,2,3
1,2.0,3.0,5,NaN


Let's generate a random series with one missing value

In [24]:
seed(10)
dates=pd.date_range("2020-10-01", periods=5, freq='D')
close=pd.Series((gauss(0,20),gauss(0,20),gauss(0,20),np.nan,gauss(0,20)), index=dates)
close

2020-10-01   -19.074340
2020-10-02    -9.181883
2020-10-03   -11.984989
2020-10-04          NaN
2020-10-05    -6.402848
Freq: D, dtype: float64

Let's have a look at the fillna() function

In [25]:
close.fillna(0)

2020-10-01   -19.074340
2020-10-02    -9.181883
2020-10-03   -11.984989
2020-10-04     0.000000
2020-10-05    -6.402848
Freq: D, dtype: float64

In [26]:
close.fillna(method="ffill")

2020-10-01   -19.074340
2020-10-02    -9.181883
2020-10-03   -11.984989
2020-10-04   -11.984989
2020-10-05    -6.402848
Freq: D, dtype: float64

In [27]:
close.fillna(method="bfill")

2020-10-01   -19.074340
2020-10-02    -9.181883
2020-10-03   -11.984989
2020-10-04    -6.402848
2020-10-05    -6.402848
Freq: D, dtype: float64

## Indexing data

Let's start by creating a data fram with multiple indexes

In [28]:
index = [('Stock A', 2018), ('Stock A', 2019),
        ('Stock B', 2018), ('Stock B', 2019),
        ('Stock C', 2018), ('Stock C', 2019)]

perf= [2.5, 7.6,
      -1.3, 9.2,
      3.4, 6.3]

stocks=pd.Series(perf, index=index)

stocks

(Stock A, 2018)    2.5
(Stock A, 2019)    7.6
(Stock B, 2018)   -1.3
(Stock B, 2019)    9.2
(Stock C, 2018)    3.4
(Stock C, 2019)    6.3
dtype: float64

In [30]:
index=pd.MultiIndex.from_tuples(index)
index

MultiIndex([('Stock A', 2018),
            ('Stock A', 2019),
            ('Stock B', 2018),
            ('Stock B', 2019),
            ('Stock C', 2018),
            ('Stock C', 2019)],
           )

In [31]:
stocks=stocks.reindex(index)
stocks

Stock A  2018    2.5
         2019    7.6
Stock B  2018   -1.3
         2019    9.2
Stock C  2018    3.4
         2019    6.3
dtype: float64

Once this is done, you can unstack or you can stack it

In [32]:
stocks_unstack=stocks.unstack(level=1)
stocks_unstack

,2018,2019
Stock A,2.5,7.6
Stock B,-1.3,9.2
Stock C,3.4,6.3


In [33]:
stocks_restack=stocks_unstack.stack()
stocks_restack

Stock A  2018    2.5
         2019    7.6
Stock B  2018   -1.3
         2019    9.2
Stock C  2018    3.4
         2019    6.3
dtype: float64

Let's add a new column to our data frame

In [35]:
stocks_df=pd.DataFrame({'total':stocks, 'div':[1.2, 1.5,2.4,2.7,0.2,0.3]})
stocks_df

total  div
Stock A 2018    2.5  1.2
        2019    7.6  1.5
Stock B 2018   -1.3  2.4
        2019    9.2  2.7
Stock C 2018    3.4  0.2
        2019    6.3  0.3

You can implement simple calculations

In [36]:
stocks_exdiv=stocks_df['total']-stocks_df['div']
stocks_exdiv.unstack()

,2018,2019
Stock A,1.3,6.1
Stock B,-3.7,6.5
Stock C,3.2,6.0


## Concatenate, merge, append etc...

In [37]:
s1=stocks_exdiv.unstack()
s1

,2018,2019
Stock A,1.3,6.1
Stock B,-3.7,6.5
Stock C,3.2,6.0


In [41]:
index= [('Stock D', 2018), ('Stock D',2017),
      ('Stock E', 2018), ('Stock E',2017)]

perf=[2.3, 0.2,
     -1.1, 1.2]

s2=pd.Series(perf, index=index)
index=pd.MultiIndex.from_tuples(index)
s2=s2.reindex(index)
s2=s2.unstack(level=1)
s2

,2017,2018
Stock D,0.2,2.3
Stock E,1.2,-1.1


In [42]:
pd.concat([s1,s2])

,2017,2018,2019
Stock A,NaN,1.3,6.1
Stock B,NaN,-3.7,6.5
Stock C,NaN,3.2,6.0
Stock D,0.2,2.3,NaN
Stock E,1.2,-1.1,NaN


In [43]:
pd.concat([s1,s2], join="inner")

,2018
Stock A,1.3
Stock B,-3.7
Stock C,3.2
Stock D,2.3
Stock E,-1.1


In [45]:
df1= pd.DataFrame({'stock':['Company A','Company B','Company C','Company D','Company E'],
                  'sector':['Utilities','Oil & Gas','Oil & Gas','Financial','Consumer']})

df1

,stock,sector
0,Company A,Utilities
1,Company B,Oil & Gas
2,Company C,Oil & Gas
3,Company D,Financial
4,Company E,Consumer


In [46]:
df2= pd.DataFrame({'stock':['Company A','Company B','Company C','Company D','Company E'],
                  'perf':[1.2,3.4,-2.3,4.7,7.6]})

df2

,stock,perf
0,Company A,1.2
1,Company B,3.4
2,Company C,-2.3
3,Company D,4.7
4,Company E,7.6


In [47]:
df3 = pd.merge(df1,df2)
df3

,stock,sector,perf
0,Company A,Utilities,1.2
1,Company B,Oil & Gas,3.4
2,Company C,Oil & Gas,-2.3
3,Company D,Financial,4.7
4,Company E,Consumer,7.6


In [48]:
df4= pd.DataFrame({'sector':['Utilities','Oil & Gas','Financial','Consumer'],
                  'style':['Defensive','Sensitive','Cyclical','Cyclical']})

df4

,sector,style
0,Utilities,Defensive
1,Oil & Gas,Sensitive
2,Financial,Cyclical
3,Consumer,Cyclical


In [49]:
df5 = pd.merge(df3,df4)
df5

,stock,sector,perf,style
0,Company A,Utilities,1.2,Defensive
1,Company B,Oil & Gas,3.4,Sensitive
2,Company C,Oil & Gas,-2.3,Sensitive
3,Company D,Financial,4.7,Cyclical
4,Company E,Consumer,7.6,Cyclical



# Let's get started with visualization !

In [50]:
#for viz
from __future__ import division

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
%matplotlib inline

#For Yahoo finance data
from pandas_datareader import data as pdr



Let's get some stock price data

In [51]:
startdate=dt(2020,1,1)
enddate=dt(2020,10,1)

tickers=['TSLA','MSFT','NFLX','BABA']

df= pdr.get_data_yahoo(tickers, start=startdate, end=enddate)['Adj Close']
df

Symbols,TSLA,MSFT,NFLX,BABA
Date,,,,
2020-01-02,86.052002,159.352386,329.809998,219.770004
2020-01-03,88.601997,157.368179,325.899994,217.000000
2020-01-06,90.307999,157.774948,335.829987,216.639999
2020-01-07,93.811996,156.336395,330.750000,217.630005
2020-01-08,98.428001,158.826569,339.260010,218.000000
...,...,...,...,...
2020-09-25,407.339996,207.820007,482.880005,271.089996
2020-09-28,421.200012,209.440002,490.649994,276.010010
2020-09-29,419.070007,207.259995,493.480011,276.929993
